# Understanding Indian EcoPolitical Growth

**Config-driven ETL over 3 public data sources · R · Shiny · plotly · shinylive (WebAssembly)**

## Hero Overview

A config-driven analytics pipeline and dashboard tracing India's economic and governance trajectory from 1789 to 2031. One CSV declares 64 indicators across 11 categories; four batch R scripts pull them from the World Bank, IMF WEO and V-Dem, unify three incompatible response shapes into a single long-format parquet table, and validate the result before it ever reaches the app.

The front end is a Shiny dashboard of 14 charts across 5 domain tabs, compiled to WebAssembly with shinylive so it runs entirely in the browser on a static host — no R server, no runtime cost. Every headline finding under a chart is computed from the data currently in view rather than written into the page.

In [1]:
outcomes = {
    "Indicators Unified": "64 across 3 sources",
    "Observations": "4,355 rows \u00b7 1789\u20132031",
    "World Bank Pull": "49 series in 2 requests (0.72s)",
    "Charts Shipped": "14 across 5 tabs",
    "Data Quality Checks": "6, sparse-aware",
    "Hosting Cost": "$0 \u2014 static WASM"
}

pd.DataFrame(outcomes.items(), columns=["Metric", "Value"])

Metric,Value
Indicators Unified,64 across 3 sources
Observations,"4,355 rows · 1789–2031"
World Bank Pull,49 series in 2 requests (0.72s)
Charts Shipped,14 across 5 tabs
Data Quality Checks,"6, sparse-aware"
Hosting Cost,$0 — static WASM


In [2]:
# Radar (skill intensity) + Bubble cluster (implementation strength).
fig

Implementation Strength by System

In [3]:
# Technology Capability Network — concepts ↔ tools (bipartite, D3).

Technology Capability Network 
 Systems delivered, the ideas behind them, and the tools they run on.

## Context

Most India dashboards stop at GDP, which answers the easy question and hides the interesting one: whether prosperity and governance moved together. Pairing them means reconciling three data providers that agree on nothing — the World Bank's REST API, IMF WEO over SDMX, and V-Dem as a bulk academic archive — each with different codes, shapes, year ranges and failure behaviour. The harder constraint was honesty: sparse public data invites charts that imply more than the numbers support.

## Workflow

One CSV declares the indicators → four batch R scripts fetch and reduce each source → a merge step unifies them into a single long-format parquet table → six checks validate it and render a report → the Shiny app reads only the finished parquet and never touches an API at runtime. The whole pipeline runs end-to-end in about six seconds from a clean tree.

## Technical Implementation

- Config contract: 64 indicators over 11 categories in one CSV, read through a single loader so the fetch loop, the join and the dashboard controls cannot drift apart.
- World Bank: 49 series batched into 2 semicolon-delimited requests against the sources endpoint, gzipped, with retry; a source_param column routes governance indicators to the separate database that otherwise returns 200 with an empty body.
- IMF WEO: SDMX key built as COUNTRY.INDICATOR.FREQUENCY — any other order returns 200 with zero series — and parsed as XML, which the API sends regardless of the Accept header.
- V-Dem: a 34MB archive expanding to ~1GB across 3,868 variables, pulled once offline and cut to 8 indices before persisting, so the app never carries the cost.
- Silent-failure guard: every adapter runs the same completeness assertion, because all three sources answer a subtly wrong request with 200 and missing series rather than an error.
- Storage: long format, not wide — 64 indicators with different year ranges would otherwise be 64 sparse columns and a per-chart drop_na. The forecast flag and the indicator metadata table are both derived, never hand-written.
- Validation: six checks with a deliberate fail/warn split, because sparsity in survey-cadence series is the data, not a defect.
- Dashboard: 14 plotly charts over 5 domain tabs, each with a method note, a caveat, and a finding line recomputed from the visible range.
- Deployment: compiled to WebAssembly via shinylive onto GitHub Pages, which required replacing arrow with nanoparquet and defeating a plotly dependency-load race with a self-terminating DOM watcher.

## Outcomes

A reproducible pipeline turning three mismatched public sources into one validated 4,355-row analytical dataset, and an interactive 14-chart dashboard that runs entirely in the browser at zero hosting cost. Each chart form was chosen against a specific way it could mislead — no dual axes across incomparable scales, no stacked area over shares that do not sum to 100%, no hardcoded findings that go stale when the range changes.